<a href="https://colab.research.google.com/github/SATHRAMCHARAN/CSA6301--THREAT-INTELLIGENCE-AND-NETWORK-SECURITY/blob/main/exp28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import hashlib

# Sample file content (simulated PE file)
sample_file_content = (
    b"MZ\x90\x00...fake_pe_header..."
    b"CreateRemoteThread..."
    b"WriteProcessMemory..."
)

# Suspicious API keywords
suspicious_api_keywords = [
    b"CreateRemoteThread",
    b"WriteProcessMemory",
    b"RegSetValue",
    b"InternetOpenUrl"
]

# Static Analysis
def static_analysis(file_bytes):
    file_hash = hashlib.sha256(file_bytes).hexdigest()

    found_apis = [
        api.decode()
        for api in suspicious_api_keywords
        if api in file_bytes
    ]

    return {
        "sha256": file_hash,
        "suspicious_apis_found": found_apis,
        "static_risk_score": len(found_apis)
    }

# Dynamic Analysis
def dynamic_analysis(sandbox_log):
    indicators = 0
    reasons = []

    if sandbox_log.get("registry_writes", 0) > 0:
        indicators += 1
        reasons.append("Modified registry")

    if sandbox_log.get("network_connections", 0) > 0:
        indicators += 1
        reasons.append("Made outbound network connection")

    if sandbox_log.get("new_processes_spawned", 0) > 1:
        indicators += 1
        reasons.append("Spawned multiple child processes")

    return {
        "dynamic_risk_score": indicators,
        "reasons": reasons
    }

# Classification
def classify(static_result, dynamic_result, threshold=2):
    total_score = (
        static_result["static_risk_score"] +
        dynamic_result["dynamic_risk_score"]
    )

    return "malicious" if total_score >= threshold else "benign"

# Malware sample
static_result = static_analysis(sample_file_content)

sandbox_log = {
    "registry_writes": 3,
    "network_connections": 1,
    "new_processes_spawned": 2
}

dynamic_result = dynamic_analysis(sandbox_log)

verdict = classify(static_result, dynamic_result)

# Benign sample
benign_static = static_analysis(
    b"just a normal text file with no suspicious content"
)

benign_dynamic = dynamic_analysis({
    "registry_writes": 0,
    "network_connections": 0,
    "new_processes_spawned": 0
})

benign_verdict = classify(benign_static, benign_dynamic)

# Display Results
print("Static Analysis:")
print(static_result)

print("\nDynamic Analysis:")
print(dynamic_result)

print("\nMalware Verdict:")
print(verdict)

print("\nBenign File Verdict:")
print(benign_verdict)

Static Analysis:
{'sha256': '25329da60625b452686ec727bb3d0f706b638afbd6fcfda5720437ab56030222', 'suspicious_apis_found': ['CreateRemoteThread', 'WriteProcessMemory'], 'static_risk_score': 2}

Dynamic Analysis:
{'dynamic_risk_score': 3, 'reasons': ['Modified registry', 'Made outbound network connection', 'Spawned multiple child processes']}

Malware Verdict:
malicious

Benign File Verdict:
benign
